# Lecture 3 — `if`, `else`, Comparisons, and Metric Decisions

Today’s goal:

> You understand how C++ makes decisions.

This is the heart of metric code.

A metric is basically many decisions like:

```text
Is trajectory empty?
Is speed too high?
Is TTC too low?
Is collision detected?
Is route handler missing?
```

In C++, these are written with:

```cpp
if
else
else if
```

## 1. Basic `if`

The simplest form:

```cpp
if (condition) {
  // do something
}
```

In [ ]:
#include <iostream>

{
  double speed_mps = 12.0;

  if (speed_mps > 10.0) {
    std::cout << "speed too high" << std::endl;
  }
}

If you change `speed_mps` to `8.0`, there is no output, because the condition is false.

## 2. Comparison operators

| Operator | Meaning | Example |
|---|---|---|
| `>` | greater than | `speed > 10.0` |
| `<` | less than | `ttc < 2.0` |
| `>=` | greater than or equal | `score >= 0.5` |
| `<=` | less than or equal | `distance <= 1.0` |
| `==` | equal | `score == 1.0` |
| `!=` | not equal | `reason != "available"` |

Important: equality uses **two equals**:

```cpp
score == 1.0
```

Assignment uses **one equals**:

```cpp
score = 1.0;
```

In [ ]:
#include <iostream>

{
  double score = 1.0;

  std::cout << std::boolalpha;
  std::cout << "score == 1.0: " << (score == 1.0) << std::endl;
  std::cout << "score != 0.0: " << (score != 0.0) << std::endl;
}

## 3. `if` + `else`

Use `else` when you want two branches.

In [ ]:
#include <iostream>

{
  double speed_mps = 8.0;

  if (speed_mps > 10.0) {
    std::cout << "speed too high" << std::endl;
  } else {
    std::cout << "speed ok" << std::endl;
  }
}

## 4. Metric-style `if` / `else`

Rule:

```text
If speed > 10:
    score = 0
    reason = "speed_too_high"
Otherwise:
    score = 1
    reason = "available"
```

In [ ]:
#include <iostream>
#include <string>

{
  const double speed_limit_mps = 10.0;
  double ego_speed_mps = 12.0;

  double score = 0.0;
  std::string reason = "unavailable";

  if (ego_speed_mps > speed_limit_mps) {
    score = 0.0;
    reason = "speed_too_high";
  } else {
    score = 1.0;
    reason = "available";
  }

  std::cout << "score = " << score << std::endl;
  std::cout << "reason = " << reason << std::endl;
}

## 5. `else if`

Use `else if` when there are multiple cases.

In [ ]:
#include <iostream>
#include <string>

{
  double ttc_s = 1.5;
  std::string status;

  if (ttc_s < 1.0) {
    status = "dangerous";
  } else if (ttc_s < 2.0) {
    status = "warning";
  } else {
    status = "safe";
  }

  std::cout << "status = " << status << std::endl;
}

## 6. Important: order matters

If `ttc_s = 0.5`, this order is wrong:

```cpp
if (ttc_s < 2.0) {
  std::cout << "warning" << std::endl;
} else if (ttc_s < 1.0) {
  std::cout << "dangerous" << std::endl;
}
```

Because `0.5 < 2.0` is already true.

Correct order:

```cpp
if (ttc_s < 1.0) {
  std::cout << "dangerous" << std::endl;
} else if (ttc_s < 2.0) {
  std::cout << "warning" << std::endl;
}
```

Always check the most severe/specific condition first.

In [ ]:
#include <iostream>
#include <string>

{
  double ttc_s = 0.5;
  std::string status;

  if (ttc_s < 1.0) {
    status = "dangerous";
  } else if (ttc_s < 2.0) {
    status = "warning";
  } else {
    status = "safe";
  }

  std::cout << "status = " << status << std::endl;
}

## 7. Logical operators: `&&`, `||`, `!`

### `&&` means AND

```cpp
if (speed_mps > 10.0 && distance_m < 5.0) {
  std::cout << "dangerous" << std::endl;
}
```

Both must be true.

### `||` means OR

```cpp
if (trajectory_empty || route_missing) {
  std::cout << "unavailable" << std::endl;
}
```

At least one must be true.

### `!` means NOT

```cpp
if (!available) {
  std::cout << "metric unavailable" << std::endl;
}
```

This means `available` is false.

In [ ]:
#include <iostream>

{
  double speed_mps = 12.0;
  double distance_m = 3.0;
  bool available = false;

  std::cout << std::boolalpha;
  std::cout << "high speed AND short distance: " << (speed_mps > 10.0 && distance_m < 5.0) << std::endl;
  std::cout << "not available: " << (!available) << std::endl;
}

## 8. Metric-style example with multiple conditions

Rule:

```text
If trajectory is empty:
    available = false
    reason = "unavailable_empty_trajectory"
Else if speed > speed limit:
    available = true
    score = 0
    reason = "speed_too_high"
Else:
    available = true
    score = 1
    reason = "available"
```

In [ ]:
#include <iostream>
#include <string>

{
  bool trajectory_empty = false;
  const double speed_limit_mps = 10.0;
  double ego_speed_mps = 12.0;

  double score = 0.0;
  bool available = false;
  std::string reason = "unavailable";

  if (trajectory_empty) {
    available = false;
    score = 0.0;
    reason = "unavailable_empty_trajectory";
  } else if (ego_speed_mps > speed_limit_mps) {
    available = true;
    score = 0.0;
    reason = "speed_too_high";
  } else {
    available = true;
    score = 1.0;
    reason = "available";
  }

  std::cout << std::boolalpha;
  std::cout << "score = " << score << std::endl;
  std::cout << "available = " << available << std::endl;
  std::cout << "reason = " << reason << std::endl;
}

## 9. Early return thinking

Metric code often uses **early return**:

```cpp
if (bad_input) {
  return result;
}

// real calculation here
```

Why?

Without early return:

```cpp
if (!trajectory_empty) {
  if (route_available) {
    if (objects_available) {
      // real metric logic here
    }
  }
}
```

This becomes a pyramid.

With early return:

```cpp
if (trajectory_empty) {
  return result;
}

if (!route_available) {
  return result;
}

if (!objects_available) {
  return result;
}

// real metric logic here
```

Much cleaner.

## 10. Available vs failed

This distinction matters a lot:

```text
available = false
→ metric could not be computed

available = true, score = 0
→ metric was computed and failed
```

Tattoo it on your debugging brain.

In [ ]:
#include <iostream>
#include <string>

{
  bool has_ttc_value = true;
  const double ttc_threshold_s = 2.0;
  double ttc_s = 1.5;

  double score = 0.0;
  bool available = false;
  std::string reason = "unavailable";

  if (!has_ttc_value) {
    reason = "unavailable_no_ttc_value";
  } else {
    available = true;
    score = 1.0;
    reason = "available";

    if (ttc_s < ttc_threshold_s) {
      score = 0.0;
      reason = "ttc_too_low";
    }
  }

  std::cout << std::boolalpha;
  std::cout << "score = " << score << std::endl;
  std::cout << "available = " << available << std::endl;
  std::cout << "reason = " << reason << std::endl;
}

## 11. Homework: lane-keeping decision

Rule:

```text
If has_lane_info is false:
    available = false
    score = 0
    reason = "unavailable_no_lane_info"

Else if lateral_deviation_m > 0.5:
    available = true
    score = 0
    reason = "lane_deviation_too_large"

Else:
    available = true
    score = 1
    reason = "available"
```

Edit the next cell.

In [ ]:
#include <iostream>
#include <string>

{
  bool has_lane_info = true;
  double lateral_deviation_m = 0.7;
  const double max_lateral_deviation_m = 0.5;

  double score = 0.0;
  bool available = false;
  std::string reason = "unavailable";

  // TODO: write your if / else logic here.

  std::cout << std::boolalpha;
  std::cout << "Metric: lane_keeping" << std::endl;
  std::cout << "lateral_deviation_m = " << lateral_deviation_m << std::endl;
  std::cout << "score = " << score << std::endl;
  std::cout << "available = " << available << std::endl;
  std::cout << "reason = " << reason << std::endl;
}

### Solution

In [ ]:
#include <iostream>
#include <string>

{
  bool has_lane_info = true;
  double lateral_deviation_m = 0.7;
  const double max_lateral_deviation_m = 0.5;

  double score = 0.0;
  bool available = false;
  std::string reason = "unavailable";

  if (!has_lane_info) {
    available = false;
    score = 0.0;
    reason = "unavailable_no_lane_info";
  } else if (lateral_deviation_m > max_lateral_deviation_m) {
    available = true;
    score = 0.0;
    reason = "lane_deviation_too_large";
  } else {
    available = true;
    score = 1.0;
    reason = "available";
  }

  std::cout << std::boolalpha;
  std::cout << "score = " << score << std::endl;
  std::cout << "available = " << available << std::endl;
  std::cout << "reason = " << reason << std::endl;
}

Next lecture: **functions** — this is where we stop writing everything inside one block and start making reusable metric calculators.